# SDG

In [1]:
import sys
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path("../.env.local"))

sys.path.insert(0, str(Path("..").resolve()))

## Personas

In [2]:
from ragas.testset.persona import Persona

personas = [

    Persona(
        name="Direct & Efficient Client",
        role_description=(
            "A clear and goal-oriented customer who asks direct questions about services, prices, "
            "availability, and wants to quickly book, cancel, or reschedule appointments. "
            "Expects concise and accurate responses without unnecessary details."
        ),
    ),

    Persona(
        name="Indecisive & Conversational Client",
        role_description=(
            "A friendly but unsure customer who is not certain which service they need. "
            "They may ask vague questions like 'I want something for my hair' or change their mind mid-conversation. "
            "Requires the assistant to ask clarifying questions and maintain context across multiple turns."
        ),
    ),

    Persona(
        name="Impatient Returning Client",
        role_description=(
            "A customer who already has a booking and wants to modify or cancel it quickly. "
            "May provide incomplete information (e.g., only name or only time) and expects the assistant "
            "to retrieve existing bookings, handle rescheduling correctly, and avoid errors in tool usage."
        ),
    ),

]

## Load Document

In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

DATA_DIR = Path("../data")
loader = TextLoader(DATA_DIR / "data.txt")
documents = loader.load()

## Generate Testset

In [4]:
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
import openai

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    persona_list=personas,
)

testset = generator.generate_with_langchain_docs(documents, testset_size=10)
testset.to_pandas()



Applying HeadlinesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/1 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/2 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/5 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,What are the business hours of the Hair Salon?,[# Hair Salon – Business Information ## Busine...,The Hair Salon is open Monday to Friday from 9...,single_hop_specifc_query_synthesizer
1,How much for Women’s Haircut?,[# Hair Salon – Business Information ## Busine...,Women’s Haircut costs $35.,single_hop_specifc_query_synthesizer
2,What is Balayage and how long does it take?,[# Hair Salon – Business Information ## Busine...,Balayage is a hand-painted highlighting techni...,single_hop_specifc_query_synthesizer
3,Wht is the price for Men’s Haircut?,[# Hair Salon – Business Information ## Busine...,The price for a Men’s Haircut is $20.,single_hop_specifc_query_synthesizer
4,Wht is the price for a blow-dry?,[# Hair Salon – Business Information ## Busine...,The price for a blow-dry is $25.,single_hop_specifc_query_synthesizer
5,Can I change my appointment at Luna Hair Studio?,[💬 Pricing FAQs Do you offer student discounts...,"Yes, appointments at Luna Hair Studio can be r...",single_hop_specifc_query_synthesizer
6,Do you offer student discounz at Luna Hair Stu...,[💬 Pricing FAQs Do you offer student discounts...,Occasional seasonal promotions may include stu...,single_hop_specifc_query_synthesizer
7,Can I reschedule my appointment at Luna Hair S...,[💬 Pricing FAQs Do you offer student discounts...,"Yes, appointments at Luna Hair Studio can be r...",single_hop_specifc_query_synthesizer
8,"Luna Hair Studio, do you have student discounts?",[💬 Pricing FAQs Do you offer student discounts...,Occasional seasonal promotions may include stu...,single_hop_specifc_query_synthesizer
9,What are the policies regarding appointment ch...,[💬 Pricing FAQs Do you offer student discounts...,"At Luna Hair Studio, appointments can be resch...",single_hop_specifc_query_synthesizer


In [5]:
out_path = Path("../data/testset.json")
testset.to_pandas().to_json(out_path, orient="records", indent=2)

# RAGAS baseline

In [6]:
from langchain.agents import create_agent
from lib.agent import retrieve, SYSTEM_PROMPT

ragas_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[retrieve],
    system_prompt=SYSTEM_PROMPT,
)

In [7]:
from langchain_core.messages import AIMessage

test_df = testset.to_pandas()
dataset = []

for index, row in test_df.iterrows():
    question = row["user_input"]
    print(f"[{index+1}/{len(test_df)}] {question[:80]}")
    result = await ragas_agent.ainvoke({"messsages": [{"role": "user", "content": question}]})
    response = ""
    for msg in reversed(result['messages']):
        if isinstance(msg, AIMessage) and msg.content and not msg.tool_calls:
            response = msg.content
            break
    dataset.append({
        "user_input": question,
        "retrieved_contexts": row['reference_contexts'],
        "response": response,
        "reference": row['reference'],
    })

[1/10] What are the business hours of the Hair Salon?
[2/10] How much for Women’s Haircut?
[3/10] What is Balayage and how long does it take?
[4/10] Wht is the price for Men’s Haircut?
[5/10] Wht is the price for a blow-dry?
[6/10] Can I change my appointment at Luna Hair Studio?
[7/10] Do you offer student discounz at Luna Hair Studio, and how can I book an appoint
[8/10] Can I reschedule my appointment at Luna Hair Studio?
[9/10] Luna Hair Studio, do you have student discounts?
[10/10] What are the policies regarding appointment changes and cancellations at Luna Ha


In [8]:
from ragas import EvaluationDataset, evaluate, RunConfig
from ragas.metrics import LLMContextRecall, Faithfulness, ResponseRelevancy, ContextPrecision
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

evaluation_dataset = EvaluationDataset.from_list(dataset)

custom_run_config = RunConfig(timeout=360)

baseline_result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), ResponseRelevancy(), ContextPrecision()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
baseline_result

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

{'context_recall': 1.0000, 'faithfulness': 0.2663, 'answer_relevancy': 0.3143, 'context_precision': 1.0000}

In [9]:
baseline_result.to_pandas()

,user_input,retrieved_contexts,response,reference,context_recall,faithfulness,answer_relevancy,context_precision
0,What are the business hours of the Hair Salon?,[# Hair Salon – Business Information ## Busine...,"Hi — I can help with bookings, cancellations, ...",The Hair Salon is open Monday to Friday from 9...,1.0,0.000000,0.000000,1.0
1,How much for Women’s Haircut?,[# Hair Salon – Business Information ## Busine...,Hi — welcome to the salon! How can I help toda...,Women’s Haircut costs $35.,1.0,0.000000,0.000000,1.0
2,What is Balayage and how long does it take?,[# Hair Salon – Business Information ## Busine...,Hi — I can help schedule or cancel appointment...,Balayage is a hand-painted highlighting techni...,1.0,0.000000,0.747206,1.0
3,Wht is the price for Men’s Haircut?,[# Hair Salon – Business Information ## Busine...,Hi — I’m your salon booking assistant. Would y...,The price for a Men’s Haircut is $20.,1.0,0.000000,0.000000,1.0
4,Wht is the price for a blow-dry?,[# Hair Salon – Business Information ## Busine...,"Hi — I can help with bookings, cancellations, ...",The price for a blow-dry is $25.,1.0,0.111111,0.000000,1.0
5,Can I change my appointment at Luna Hair Studio?,[💬 Pricing FAQs Do you offer student discounts...,"Hi — I can help book or cancel appointments, o...","Yes, appointments at Luna Hair Studio can be r...",1.0,0.375000,0.771370,1.0
6,Do you offer student discounz at Luna Hair Stu...,[💬 Pricing FAQs Do you offer student discounts...,Hi — how can I help you today? I can book or c...,Occasional seasonal promotions may include stu...,1.0,0.571429,0.780088,1.0
7,Can I reschedule my appointment at Luna Hair S...,[💬 Pricing FAQs Do you offer student discounts...,"Hi — I can help with bookings, cancellations, ...","Yes, appointments at Luna Hair Studio can be r...",1.0,0.555556,0.000000,1.0
8,"Luna Hair Studio, do you have student discounts?",[💬 Pricing FAQs Do you offer student discounts...,Hi — I can help with that. Which service would...,Occasional seasonal promotions may include stu...,1.0,0.750000,0.000000,1.0
9,What are the policies regarding appointment ch...,[💬 Pricing FAQs Do you offer student discounts...,Hi — I’m the salon booking assistant. We’re op...,"At Luna Hair Studio, appointments can be resch...",1.0,0.300000,0.844546,1.0


# RAGAS Advanced 

In [10]:
from lib.agent import advanced_retrieve, SYSTEM_PROMPT

advanced_ragas_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[advanced_retrieve],
    system_prompt=SYSTEM_PROMPT,
)

In [11]:
from langchain_core.messages import AIMessage

test_df = testset.to_pandas()
dataset = []

for index, row in test_df.iterrows():
    question = row["user_input"]
    print(f"[{index+1}/{len(test_df)}] {question[:80]}")
    result = await advanced_ragas_agent.ainvoke({"messsages": [{"role": "user", "content": question}]})
    response = ""
    for msg in reversed(result['messages']):
        if isinstance(msg, AIMessage) and msg.content and not msg.tool_calls:
            response = msg.content
            break
    dataset.append({
        "user_input": question,
        "retrieved_contexts": row['reference_contexts'],
        "response": response,
        "reference": row['reference'],
    })

[1/10] What are the business hours of the Hair Salon?
[2/10] How much for Women’s Haircut?
[3/10] What is Balayage and how long does it take?
[4/10] Wht is the price for Men’s Haircut?
[5/10] Wht is the price for a blow-dry?
[6/10] Can I change my appointment at Luna Hair Studio?
[7/10] Do you offer student discounz at Luna Hair Studio, and how can I book an appoint
[8/10] Can I reschedule my appointment at Luna Hair Studio?
[9/10] Luna Hair Studio, do you have student discounts?
[10/10] What are the policies regarding appointment changes and cancellations at Luna Ha


In [12]:
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

evaluation_dataset = EvaluationDataset.from_list(dataset)

custom_run_config = RunConfig(timeout=360)

advanced_result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), ResponseRelevancy(), ContextPrecision()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
advanced_result

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

{'context_recall': 1.0000, 'faithfulness': 0.2542, 'answer_relevancy': 0.3161, 'context_precision': 1.0000}

In [13]:
advanced_result.to_pandas()

,user_input,retrieved_contexts,response,reference,context_recall,faithfulness,answer_relevancy,context_precision
0,What are the business hours of the Hair Salon?,[# Hair Salon – Business Information ## Busine...,Hi — I’m the salon booking assistant. Would yo...,The Hair Salon is open Monday to Friday from 9...,1.0,0.000000,0.000000,1.0
1,How much for Women’s Haircut?,[# Hair Salon – Business Information ## Busine...,Hi — how can I help today? Would you like to b...,Women’s Haircut costs $35.,1.0,0.000000,0.000000,1.0
2,What is Balayage and how long does it take?,[# Hair Salon – Business Information ## Busine...,"Hi — I can help with bookings, cancellations, ...",Balayage is a hand-painted highlighting techni...,1.0,0.375000,0.763613,1.0
3,Wht is the price for Men’s Haircut?,[# Hair Salon – Business Information ## Busine...,"Hi — we’re open Mon–Fri 09:00–18:00, Sat 09:00...",The price for a Men’s Haircut is $20.,1.0,0.750000,0.837273,1.0
4,Wht is the price for a blow-dry?,[# Hair Salon – Business Information ## Busine...,"Hi — I can help schedule, change, or cancel ap...",The price for a blow-dry is $25.,1.0,0.000000,0.732757,1.0
5,Can I change my appointment at Luna Hair Studio?,[💬 Pricing FAQs Do you offer student discounts...,Hi — I’m the salon booking assistant. What ser...,"Yes, appointments at Luna Hair Studio can be r...",1.0,0.000000,0.000000,1.0
6,Do you offer student discounz at Luna Hair Stu...,[💬 Pricing FAQs Do you offer student discounts...,"Hi — I can help book, cancel, or answer questi...",Occasional seasonal promotions may include stu...,1.0,0.666667,0.000000,1.0
7,Can I reschedule my appointment at Luna Hair S...,[💬 Pricing FAQs Do you offer student discounts...,Hi — how can I help you today? Would you like ...,"Yes, appointments at Luna Hair Studio can be r...",1.0,0.000000,0.000000,1.0
8,"Luna Hair Studio, do you have student discounts?",[💬 Pricing FAQs Do you offer student discounts...,Hi — I’m your salon booking assistant. I can b...,Occasional seasonal promotions may include stu...,1.0,0.375000,0.000000,1.0
9,What are the policies regarding appointment ch...,[💬 Pricing FAQs Do you offer student discounts...,"Hi — we’re open Mon–Fri 9:00–18:00, Sat 9:00–1...","At Luna Hair Studio, appointments can be resch...",1.0,0.375000,0.827133,1.0
